# Next-Word Prediction with LSTM and Additive Attention

**Dataset:** The Adventures of Sherlock Holmes by Arthur Conan Doyle, Project Gutenberg UTF-8 text.

This notebook is the research-facing companion to the production scripts in `src/`. It covers EDA, preprocessing, model architecture, Bahdanau-style additive attention, training, evaluation, text generation, top-5 probability outputs, and attention visualization.

## Assignment Checklist

- Automatic dataset download and loading
- Text cleaning, tokenization, and sequence generation
- Embedding -> Bidirectional LSTM -> Attention -> Dense -> Dropout -> Softmax model
- Training with epoch-wise loss, top-1 accuracy, top-5 accuracy, and perplexity
- Train/validation/test evaluation artifacts
- Text generation with temperature, top-k sampling, nucleus sampling, and repetition penalty
- Top-5 next-word predictions with probabilities
- Attention heatmap visualization


## 1. Setup

The notebook imports the same modules used by the CLI scripts. If running from a fresh clone, install dependencies first:

```bash
uv sync
```


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from src.config import DEFAULT_CONFIG

print(f"Project root: {PROJECT_ROOT}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")


## 2. Dataset Download and EDA

The loader downloads the corpus when missing, validates that it is the expected Sherlock Holmes text, and removes Project Gutenberg boilerplate before modeling.


In [ ]:
from src.data_loader import corpus_statistics, download_corpus, load_corpus

corpus_path = download_corpus()
raw_text = load_corpus(corpus_path)
stats = corpus_statistics(raw_text)

stats


In [ ]:
print(raw_text[:800])


## 3. Preprocessing

The preprocessing pipeline lowercases text, removes chapter headings and punctuation while preserving apostrophes, fits a capped Keras tokenizer, and creates sparse-label next-word training pairs.

For each position `i` in the token stream:

`X_i = [w_i, ..., w_{i+T-1}]`

`y_i = w_{i+T}`

Using sparse integer labels avoids a large one-hot target matrix and keeps training memory practical on a laptop.


In [ ]:
from src.preprocessor import (
    MAX_VOCAB_SIZE,
    SEQUENCE_LENGTH,
    clean_text,
    decode_tokens,
    preprocess,
    summarize_vocabulary,
)

X, y, tokenizer, vocab_size = preprocess(
    raw_text,
    max_vocab=MAX_VOCAB_SIZE,
    seq_len=SEQUENCE_LENGTH,
    save_tok=True,
)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Vocabulary size including padding: {vocab_size:,}")
print("Top tokens:", summarize_vocabulary(tokenizer, n=15))


In [ ]:
sample_idx = 42
sample_context = decode_tokens(X[sample_idx], tokenizer)
sample_target = decode_tokens([y[sample_idx]], tokenizer)[0]

print("Context:", " ".join(sample_context))
print("Target :", sample_target)


## 4. Model Architecture

The assignment architecture is implemented in `src/model.py`:

```text
Input token ids
-> Embedding(vocab_size, 128)
-> Bidirectional LSTM(256, return_sequences=True)
-> Bahdanau-style additive attention
-> Dense(256, relu)
-> Dropout(0.3)
-> Dense(vocab_size, softmax)
```

The output is `P(next_word | context)` over the vocabulary, trained with sparse categorical cross-entropy.


In [ ]:
from src.model import build_model, print_model_summary

model = build_model(
    vocab_size=vocab_size,
    seq_len=SEQUENCE_LENGTH,
    embed_dim=DEFAULT_CONFIG.model.embedding_dim,
    lstm_units=DEFAULT_CONFIG.model.lstm_units,
    attention_units=DEFAULT_CONFIG.model.attention_units,
    dense_units=DEFAULT_CONFIG.model.dense_units,
    dropout_rate=DEFAULT_CONFIG.model.dropout_rate,
)

print_model_summary(model)


## 5. Attention Mechanism

This project uses Bahdanau-style additive attention over the BiLSTM hidden states.

Let the BiLSTM produce hidden states:

$$H = [h_1, h_2, ..., h_T]$$

Because next-word prediction has no separate decoder state, the layer uses an encoder-side query:

$$q = \frac{1}{T}\sum_{t=1}^{T} h_t$$

Each hidden state receives an additive attention score:

$$e_t = v^T \tanh(W_1 h_t + W_2 q)$$

Scores are normalized into attention weights:

$$\alpha_t = \frac{\exp(e_t)}{\sum_j \exp(e_j)}$$

The context vector is the weighted sum of hidden states:

$$c = \sum_{t=1}^{T} \alpha_t h_t$$

**Why attention helps:** an LSTM final state can compress away useful early context. Attention lets the classifier re-weight all hidden states, which is useful in Sherlock Holmes prose where relevant evidence, subjects, and modifiers may appear several words before the next-word target.


## 6. Training

Full training is available through the CLI and from this notebook. The default notebook setting avoids accidentally starting a long run. Set `RUN_FULL_TRAINING = True` to train here, or run:

```bash
uv run python -m src.train
```


In [ ]:
from src.train import train

RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    model, tokenizer, vocab_size, history, final_metrics = train()
    final_metrics
else:
    print("Training skipped in notebook preview. Set RUN_FULL_TRAINING = True for a full run.")
    print("CLI equivalent: uv run python -m src.train")


## 7. Training Curves and Perplexity

The training script saves epoch-wise metrics to `outputs/training_history.json` and a four-panel plot to `outputs/training_history.png`.


In [ ]:
from IPython.display import Image, display

history_plot = PROJECT_ROOT / "outputs" / "training_history.png"
if history_plot.exists():
    display(Image(filename=str(history_plot)))
else:
    print("No training plot found yet. Run training to create outputs/training_history.png.")


## 8. Evaluation

The evaluator rebuilds the deterministic test split and computes loss, top-1 accuracy, top-5 accuracy, and perplexity.

```bash
uv run python -m src.evaluate
```


In [ ]:
from src.evaluate import evaluate_saved_model

RUN_EVALUATION = False

if RUN_EVALUATION:
    metrics = evaluate_saved_model()
    metrics
else:
    metrics_path = PROJECT_ROOT / "outputs" / "metrics.json"
    if metrics_path.exists():
        print(metrics_path.read_text()[:2000])
    else:
        print("No metrics artifact found yet. Run training/evaluation to create metrics JSON.")


## 9. Text Generation and Top-5 Predictions

`generate_text()` supports:

- `temperature` for conservative versus diverse sampling
- `top_k` candidate restriction and probability reporting
- `strategy='greedy'`, `'top_k'`, `'nucleus'`, or `'multinomial'`
- repetition penalty on recently generated tokens


In [ ]:
from src.generate import SEED_PHRASES, generate_text, load_generation_artifacts, show_top5_table

try:
    trained_model, trained_tokenizer = load_generation_artifacts()
    generated, breakdown = generate_text(
        "It was a dark and stormy",
        next_words=35,
        model=trained_model,
        tokenizer=trained_tokenizer,
        temperature=0.8,
        top_k=5,
        strategy="top_k",
        random_seed=42,
    )
    print(generated)
    show_top5_table(breakdown[:6])
except RuntimeError as exc:
    print(exc)
    print("Run `uv run python -m src.train` to regenerate compatible model artifacts.")


In [ ]:
try:
    for seed in SEED_PHRASES:
        generated, _ = generate_text(
            seed,
            next_words=30,
            model=trained_model,
            tokenizer=trained_tokenizer,
            temperature=0.8,
            top_k=5,
            strategy="top_k",
            random_seed=7,
        )
        print(f"Seed: {seed}")
        print(generated)
        print()
except NameError:
    print("Load or train a compatible model before running generation examples.")


## 10. Attention Heatmap

The heatmap visualizes which context tokens received the highest attention mass before the next-word classifier.


In [ ]:
from src.generate import attention_for_seed
from src.visualization import plot_attention_heatmap

try:
    tokens, weights = attention_for_seed(
        "Holmes looked at the curious paper",
        trained_model,
        trained_tokenizer,
    )
    heatmap_path = plot_attention_heatmap(tokens, weights)
    display(Image(filename=str(heatmap_path)))
except NameError:
    print("Load or train a compatible model before plotting attention.")
except RuntimeError as exc:
    print(exc)


## 11. Qualitative Analysis Template

After training, inspect generated examples for:

- Sherlock-like vocabulary and tone
- grammatical continuity from the seed phrase
- low repetition across 30+ generated tokens
- whether top-5 candidates are semantically plausible
- whether attention focuses on content words rather than padding

For a rigorous submission, include the exact final metrics from `outputs/metrics.json` and representative generated samples from this notebook.
